In [ ]:
!pip install -q spacy

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from spacy import displacy

In [ ]:
nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully!")

spaCy model loaded successfully!


In [ ]:
text = """
Apple Inc. announced on September 1, 2026 that it will invest $5 billion
in a new technology center in California. CEO Tim Cook said the company
will create more than 10,000 jobs over the next five years.

The new center will be located near San Francisco and will focus on
artificial intelligence and semiconductor research. Microsoft and Google
are also expanding their AI operations in the United States.
"""

In [ ]:
doc = nlp(text)

print("Text processed successfully!")

Text processed successfully!


In [ ]:
print("========== NAMED ENTITIES ==========\n")

for ent in doc.ents:
    print(
        f"Entity: {ent.text:<25} "
        f"Label: {ent.label_:<10} "
        f"Description: {spacy.explain(ent.label_)}"
    )

========== NAMED ENTITIES ==========

Entity: Apple Inc.                Label: ORG        Description: Companies, agencies, institutions, etc.
Entity: September 1, 2026         Label: DATE       Description: Absolute or relative dates or periods
Entity: $5 billion                Label: MONEY      Description: Monetary values, including unit
Entity: California                Label: GPE        Description: Countries, cities, states
Entity: Tim Cook                  Label: PERSON     Description: People, including fictional
Entity: more than 10,000          Label: CARDINAL   Description: Numerals that do not fall under another type
Entity: the next five years       Label: DATE       Description: Absolute or relative dates or periods
Entity: San Francisco             Label: GPE        Description: Countries, cities, states
Entity: Microsoft                 Label: ORG        Description: Companies, agencies, institutions, etc.
Entity: Google                    Label: ORG        Description:

In [ ]:
for ent in doc.ents:
    print(ent.text, "→", ent.label_)

Apple Inc. → ORG
September 1, 2026 → DATE
$5 billion → MONEY
California → GPE
Tim Cook → PERSON
more than 10,000 → CARDINAL
the next five years → DATE
San Francisco → GPE
Microsoft → ORG
Google → ORG
AI → GPE
the United States → GPE


In [ ]:
print(spacy.explain("ORG"))
print(spacy.explain("GPE"))
print(spacy.explain("PERSON"))
print(spacy.explain("MONEY"))

Companies, agencies, institutions, etc.
Countries, cities, states
People, including fictional
Monetary values, including unit


In [ ]:
displacy.render(
    doc,
    style="ent",
    jupyter=True
)

In [ ]:
people = [
    ent.text
    for ent in doc.ents
    if ent.label_ == "PERSON"
]

print("People:")
print(people)

People:
['Tim Cook']


In [ ]:
organizations = [
    ent.text
    for ent in doc.ents
    if ent.label_ == "ORG"
]

print("Organizations:")
print(organizations)

Organizations:
['Apple Inc.', 'Microsoft', 'Google']


In [ ]:
locations = [
    ent.text
    for ent in doc.ents
    if ent.label_ in ["GPE", "LOC"]
]

print("Locations:")
print(locations)

Locations:
['California', 'San Francisco', 'AI', 'the United States']


In [ ]:
import pandas as pd

entities = []

for ent in doc.ents:
    entities.append({
        "Entity": ent.text,
        "Label": ent.label_,
        "Description": spacy.explain(ent.label_)
    })

entity_df = pd.DataFrame(entities)

display(entity_df)

,Entity,Label,Description
0,Apple Inc.,ORG,"Companies, agencies, institutions, etc."
1,"September 1, 2026",DATE,Absolute or relative dates or periods
2,$5 billion,MONEY,"Monetary values, including unit"
3,California,GPE,"Countries, cities, states"
4,Tim Cook,PERSON,"People, including fictional"
5,"more than 10,000",CARDINAL,Numerals that do not fall under another type
6,the next five years,DATE,Absolute or relative dates or periods
7,San Francisco,GPE,"Countries, cities, states"
8,Microsoft,ORG,"Companies, agencies, institutions, etc."
9,Google,ORG,"Companies, agencies, institutions, etc."


In [ ]:
TRAIN_DATA = [
    (
        "The patient was diagnosed with diabetes and prescribed Metformin.",
        {
            "entities": [
                (30, 38, "DISEASE"),
                (56, 65, "MEDICATION")
            ]
        }
    ),

    (
        "The doctor prescribed Insulin to treat diabetes.",
        {
            "entities": [
                (21, 28, "MEDICATION"),
                (41, 49, "DISEASE")
            ]
        }
    ),

    (
        "Hypertension can be treated using Amlodipine.",
        {
            "entities": [
                (0, 12, "DISEASE"),
                (33, 43, "MEDICATION")
            ]
        }
    ),

    (
        "The patient has asthma and uses Salbutamol.",
        {
            "entities": [
                (18, 24, "DISEASE"),
                (35, 45, "MEDICATION")
            ]
        }
    ),

    (
        "Doctors are studying treatments for cancer using chemotherapy.",
        {
            "entities": [
                (38, 44, "DISEASE"),
                (50, 62, "TREATMENT")
            ]
        }
    )
]

In [ ]:
import spacy

custom_nlp = spacy.blank("en")

print("Blank model created.")

Blank model created.


In [ ]:
ner = custom_nlp.add_pipe("ner")

# Add custom entity labels
ner.add_label("DISEASE")
ner.add_label("MEDICATION")
ner.add_label("TREATMENT")

print("Custom labels added.")

Custom labels added.


In [ ]:
from spacy.training import Example

optimizer = custom_nlp.begin_training()

for iteration in range(30):

    losses = {}

    for text, annotations in TRAIN_DATA:

        example = Example.from_dict(
            custom_nlp.make_doc(text),
            annotations
        )

        custom_nlp.update(
            [example],
            drop=0.3,
            losses=losses
        )

    if iteration % 5 == 0:
        print(
            f"Iteration {iteration}, "
            f"Loss: {losses}"
        )

/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "The patient was diagnosed with diabetes and prescr..." with entities "[(30, 38, 'DISEASE'), (56, 65, 'MEDICATION')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "The doctor prescribed Insulin to treat diabetes." with entities "[(21, 28, 'MEDICATION'), (41, 49, 'DISEASE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "

Iteration 0, Loss: {'ner': np.float32(25.68656)}
Iteration 5, Loss: {'ner': np.float32(1.9870881)}
Iteration 10, Loss: {'ner': np.float32(0.73894066)}
Iteration 15, Loss: {'ner': np.float32(0.5087675)}
Iteration 20, Loss: {'ner': np.float32(1.1110076e-05)}
Iteration 25, Loss: {'ner': np.float32(2.9213345e-06)}


In [ ]:
test_text = """
The patient has diabetes and hypertension.
The doctor prescribed Metformin and Amlodipine.
The patient is also receiving chemotherapy.
"""

doc_custom = custom_nlp(test_text)

for ent in doc_custom.ents:
    print(
        f"{ent.text} → {ent.label_}"
    )

In [ ]:
displacy.render(
    doc_custom,
    style="ent",
    jupyter=True
)

/usr/local/lib/python3.13/dist-packages/spacy/displacy/__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


In [ ]:
nlp = spacy.load("en_core_web_sm")

ner = nlp.get_pipe("ner")

ner.add_label("DISEASE")
ner.add_label("MEDICATION")
ner.add_label("TREATMENT")

1

In [ ]:
text = """
Russian President Vladimir Putin has been urged by Indian Prime Minister Narendra Modi, a close ally, to end Moscow's war against Ukraine.

"Every day spent in war pushes humanity back, and every step taken towards peace fills humanity with new hope and enthusiasm," Modi said on Monday.

Modi, whose country's economy depends heavily on discounted oil from Moscow, has made several similar pleas to Putin since he launched a full-scale invasion of Ukraine in 2022.

BBC correspondents in Delhi and Moscow look at what lies behind Modi's renewed appeal for peace, and Putin's response.

Vikas Pandey
India digital editor, Delhi
Modi's comments on ending the war in Ukraine are not surprising and don't suggest any tactical change in India's foreign policy.

In fact, speaking on the sidelines of a regional summit in 2022 - less than a year into the conflict - Modi told Putin publicly that "now is not the time for war".

But it would be an oversimplification to suggest that Modi's current or past statements on the war can be seen as a rebuff to Putin or Russia's strategy.

Western countries have pushed Modi hard to take a clear stand on the war - but Delhi has largely stayed neutral, urging dialogue to end the war instead.

The stand hasn't made many Western capitals happy but they have continued to court Modi and India - it's just too big and too fast-growing an economy to be ignored.

In this context, Modi's current statement cannot be seen as an attempt to please a third country.

Getty Images Storage tanks at a Bharat Petroleum Corp. oil refinery in Mumbai, India. Photo: August 2025Getty Images
India's economy relies heavily on imported Russian oil

If anything, it is very much in line with India's policy of strategic autonomy. Which then begs the question: why would Modi say something on the war so publicly?

The answer lies in the long-standing Russia-India ties - the two countries have generally found a way to work together, even when they see geopolitics through different lenses.

Moscow has continued to be one of India's strongest allies despite Delhi forging closer relationships with Europe and the US in the past three decades.

Delhi is one of the founding members of the Brics grouping of nations along with Russia and China, and it is also a member of the Shanghai Cooperation Organisation with the two countries.

But at the same time, Delhi is also a member of the Quad group with Australia, Japan and the US - Moscow has not been happy about this association.

Still, Moscow and Delhi tend to work through these differences. Moreover, Russia still supplies around 40% of India's defence imports and it has also been a steady supplier of Delhi's energy needs.

So, Modi's latest statement is not expected to cause any rift in Moscow-Delhi relations and analysts say it's unlikely that it would have come as a surprise to Putin or his diplomats.


As for the West and Modi's critics in India, they will feel that the prime minister should have been even more direct in his criticism of the war.

Steve Rosenberg
Russia editor, Moscow
This isn't the first time a foreign leader has publicly urged Putin to end the war in Ukraine.

Just five weeks ago the President of Kazakhstan Kassym-Jomart Tokayev suggested Putin freeze the conflict and return to the negotiating table.

He didn't. And he hasn't.

There's no sign of these public calls for peace changing Putin's calculations.

To Modi's call for a cessation of hostilities, the Kremlin leader replied:

"Thank you very much, prime minister. We are moving forward along this path."

Polite, but vague. And non-committal.

After all, a path can be short. But it can also be long. Very long.

And "moving forward" at what pace?

And, crucially, on which path?

All the evidence suggests that Russia's leadership remains convinced it is on a path to military victory in Ukraine. To frame that in terms of peace, "moving forward" to peace on Russia's terms dictated by military success.


Therefore, for now, the fighting goes on.

Despite the fact that, after more than four-and-a-half years of war and huge military losses, the Kremlin has yet to secure victory.

And despite Ukraine's success in bringing the war back home to Russians: for example, through long-range drone strikes on Russian oil refineries that have sparked fuel shortages and petrol queues across the country.

Reuters A woman in Ukraine's town of Boryspil, outside Kyiv, collects belongings from her flat destroyed by overnight Russian strikes. Photo: 1 September 2026Reuters
Four people were killed in overnight Russian strikes on Ukraine's town of Boryspil, outside Kyiv
In recent days Russia has stepped up its attacks on Ukraine.

"After [Russia] intensified its strikes on Ukraine's military-industrial facilities and critical infrastructure," wrote the Kommersant newspaper this morning, "the Russian side is convinced that in the military confrontation time is working against Kyiv."

While Putin believes that, he'll be reluctant to pause his so-called "special military operation".
"""

In [ ]:
doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "→", ent.label_)

Russian → NORP
Vladimir Putin → PERSON
Indian → NORP
Narendra Modi → PERSON
Moscow → GPE
Ukraine → GPE
Monday → DATE
Moscow → GPE
Putin → PERSON
Ukraine → GPE
2022 → DATE
BBC → ORG
Delhi → GPE
Moscow → GPE
Modi → GPE
Putin → PERSON
Vikas Pandey → ORG
India → GPE
Delhi
Modi's → ORG
Ukraine → GPE
India → GPE
2022 → DATE
less than a year → DATE
Putin → PERSON
Modi → PERSON
Putin → PERSON
Russia → GPE
Delhi → GPE
Western → NORP
Modi → GPE
India → GPE
Modi → PERSON
third → ORDINAL
Petroleum Corp. → ORG
Mumbai → GPE
India → GPE
August 2025Getty → DATE
Images
India's → ORG
Russian → NORP
India → GPE
Russia → GPE
India → GPE
two → CARDINAL
Moscow → GPE
India → GPE
Delhi → GPE
Europe → LOC
US → GPE
the past three decades → DATE
Delhi → GPE
Brics → GPE
Russia → GPE
China → GPE
the Shanghai Cooperation Organisation → ORG
two → CARDINAL
Delhi → GPE
Quad → ORG
Australia → GPE
Japan → GPE
US → GPE
Moscow → GPE
Moscow → GPE
Delhi → GPE
Russia → GPE
around 40% → PERCENT
India → GPE
Delhi → GPE
Modi → 

In [ ]:
displacy.render(
    doc,
    style="ent",
    jupyter=True
)